In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys; sys.path.insert(0,'../codes/')
import foftools as fof
from prob_g3groupfinder import pfof_comoving, gauss
from group_purity import get_metrics_by_group
from astropy.stats import biweight_scale, biweight_location
from copy import deepcopy
from astropy.cosmology import LambdaCDM
cosmo = LambdaCDM(70.,0.3,0.7)
from numba import njit

%matplotlib inline

In [2]:
eco = pd.read_csv("/srv/one/zhutchen/g3groupfinder/resolve_and_eco/ECOdata_G3catalog_luminosity.csv").set_index('name')
eco = eco[(eco.absrmag<=-17.33) & ((eco.g3grpngi_l+eco.g3grpndw_l)>1)]
photzdata = pd.read_csv("/srv/one/hperk4/eco_resb_decals_photoz.csv")

In [3]:
photzdata = photzdata[photzdata.name.str.startswith('ECO')].set_index('name')
eco = pd.concat([eco,photzdata],axis=1).loc[eco.index,:]

In [4]:
degradedcz = deepcopy(eco.cz.to_numpy())
zphot = deepcopy(eco.photo_z_corr.to_numpy())
zphoterr = deepcopy(eco.e_tab_corr.to_numpy())
degradedczerr = np.zeros_like(degradedcz)+35

idx = np.random.choice(np.indices(degradedcz.shape)[0], size=int(0.85*len(degradedcz)), replace=False)
degradedcz[idx] = zphot[idx]
degradedczerr[idx] = zphoterr[idx]
sel = np.isnan(degradedcz)
degradedcz[sel] = eco.cz.to_numpy()[sel]
degradedczerr[sel] = 35

In [5]:
eco.loc[:,'degradedcz'] = degradedcz
eco.loc[:,'degradedczerr'] = degradedczerr 

In [6]:
@njit
def gauss_vectorized(x, mu, sigma):
    """
    Gaussian function.
    Arguments:
        x - dynamic variable
        mu - centroid of distribution
        sigma - standard error of distribution
    Returns:
        PDF value evaluated at `x`
    """
    return 1/(np.sqrt(2*np.pi) * sigma) * np.exp(-1 * 0.5 * ((x-mu)/sigma) * ((x-mu)/sigma))

@njit
def get_median_eCDF(xx,aa):
    """
    xx : values in distribution
    aa : PDF of data (arbitrary units)
    """
    #xx=np.array(xx)
    #aa=np.array(aa)
    cs = np.cumsum(aa)
    return xx[np.argmin(np.abs(cs-0.5*cs[-1]))]
    
@njit
def prob_group_skycoords(galaxyra, galaxydec, galaxycz, galaxyczerr, galaxygrpid, return_z_pdfs=False):
    """
    -----
    Obtain a list of group centers (RA/Dec/cz) given a list of galaxy coordinates (equatorial)
    and their corresponding group ID numbers.
    
    Inputs (all same length)
       galaxyra : 1D iterable,  list of galaxy RA values in decimal degrees
       galaxydec : 1D iterable, list of galaxy dec values in decimal degrees
       galaxycz : 1D iterable, list of galaxy cz values in km/s
       galaxyczerr : 1D iterable, list of galaxy cz 
       galaxygrpid : 1D iterable, group ID number for every galaxy in previous arguments.
    
    Outputs (all shape match `galaxyra`)
       groupra : RA in decimal degrees of galaxy i's group center.
       groupdec : Declination in decimal degrees of galaxy i's group center.
       groupcz : Redshift velocity in km/s of galaxy i's group center.
    
    Note: the FoF code of AA Berlind uses theta_i = declination, with theta_cen = 
    the central declination. This version uses theta_i = pi/2-dec, with some trig functions
    changed so that the output *matches* that of Berlind's FoF code (my "deccen" is the same as
    his "thetacen", to be exact.)
    -----
    """
    # Prepare cartesian coordinates of input galaxies
    ngalaxies = len(galaxyra)
    galaxyphi = galaxyra * np.pi/180.
    galaxytheta = np.pi/2. - galaxydec*np.pi/180.
    galaxyx = np.expand_dims((np.sin(galaxytheta)*np.cos(galaxyphi)),axis=1) # equivalent to [:,np.newaxis]
    galaxyy = np.expand_dims((np.sin(galaxytheta)*np.sin(galaxyphi)),axis=1)
    galaxyz = np.expand_dims(((np.cos(galaxytheta))),axis=1)
    # Prepare output arrays
    uniqidnumbers = np.unique(galaxygrpid)
    groupra = np.zeros(ngalaxies)
    groupdec = np.zeros(ngalaxies)
    groupcz = np.zeros(ngalaxies)
    cspeed=3e5
    galaxycz = np.expand_dims(galaxycz,axis=1) / cspeed
    galaxyczerr = np.expand_dims(galaxyczerr,axis=1) / cspeed
    #zzmesh = np.arange(-2,3,1/(cspeed)) # resolution of 1 km/s
    for i,uid in enumerate(uniqidnumbers):
        sel=np.where(galaxygrpid==uid)
        if len(sel[0])==1:
            groupra[sel] = galaxyra[sel]
            groupdec[sel] = galaxydec[sel]
            groupcz[sel] = galaxycz[sel]*cspeed
        else:
            #zzmesh = np.arange(np.min(galaxycz[sel])-5*np.max(galaxyczerr[sel]), np.max(galaxycz[sel])+5*np.max(galaxyczerr[sel]), 1/cspeed)
            xmesh = np.arange(np.min(galaxycz[sel]*galaxyx[sel])-5*np.max(np.abs(galaxyczerr[sel]*galaxyx[sel])), np.max(galaxycz[sel]*galaxyx[sel])+5*np.max(np.abs(galaxyczerr[sel]*galaxyx[sel])), np.min(galaxyczerr[sel])/1000.)
            xcen = get_median_eCDF(xmesh, np.sum(gauss_vectorized(xmesh, galaxycz[sel]*galaxyx[sel], galaxyczerr[sel]*galaxyx[sel]),axis=0))
            ymesh = np.arange(np.min(galaxycz[sel]*galaxyy[sel])-5*np.max(np.abs(galaxyczerr[sel]*galaxyy[sel])), np.max(galaxycz[sel]*galaxyy[sel])+5*np.max(np.abs(galaxyczerr[sel]*galaxyy[sel])), np.min(galaxyczerr[sel])/1000.)
            ycen = get_median_eCDF(ymesh, np.sum(gauss_vectorized(ymesh, galaxycz[sel]*galaxyy[sel], galaxyczerr[sel]*galaxyy[sel]),axis=0))
            zmesh = np.arange(np.min(galaxycz[sel]*galaxyz[sel])-5*np.max(np.abs(galaxyczerr[sel]*galaxyz[sel])), np.max(galaxycz[sel]*galaxyz[sel])+5*np.max(np.abs(galaxyczerr[sel]*galaxyz[sel])), np.min(galaxyczerr[sel])/1000.)
            zcen = get_median_eCDF(zmesh, np.sum(gauss_vectorized(zmesh, galaxycz[sel]*galaxyz[sel], galaxyczerr[sel]*galaxyz[sel]),axis=0))    
            redshiftcen = np.sqrt(xcen*xcen + ycen*ycen + zcen*zcen)
            deccen = np.arcsin(zcen/redshiftcen)*180.0/np.pi # degrees
            if (ycen >=0 and xcen>=0):
                phicor = 0.0
            elif (ycen < 0 and xcen < 0):
                 phicor = 180.0
            elif (ycen >= 0 and xcen < 0):
                phicor = 180.0
            elif (ycen < 0 and xcen >=0):
                phicor = 360.0
            elif (xcen==0 and ycen==0):
                 print("Warning: xcen=0 and ycen=0 for group ", uid)
            racen=np.arctan(ycen/xcen)*(180/np.pi)+phicor # in degrees
            groupra[sel] = racen # in degrees
            groupdec[sel] = deccen # in degrees
            groupcz[sel] = cspeed * redshiftcen
    if return_z_pdfs:
        zmesh = np.arange(0,np.max(galaxycz)+8*np.max(galaxyczerr), 1/cspeed) # 1 km/s resolution
        z_pdfs = np.zeros((len(galaxygrpid), len(zmesh)))
        for i,uid in enumerate(uniqidnumbers):
            sel=np.where(galaxygrpid==uid)
            z_pdfs[i]=np.sum(gauss_vectorized(zmesh, galaxycz[sel], galaxyczerr[sel]),axis=0)
        pdfoutput = {'zmesh':zmesh, 'pdf':z_pdfs, 'grpid':galaxygrpid}
    else:
        pdfoutput=None
    return groupra, groupdec, groupcz, pdfoutput

In [ ]:
#tmpra, tmpdec, tmpcz = prob_group_skycoords(eco.radeg.to_numpy(), eco.dedeg.to_numpy(), eco.degradedcz.to_numpy(), eco.degradedczerr.to_numpy(), eco.g3grp_l.to_numpy())
tmpra, tmpdec, tmpcz, pdfs = prob_group_skycoords(eco.radeg.to_numpy(), eco.dedeg.to_numpy(), eco.cz.to_numpy(), eco.cz.to_numpy()*0 + 50, eco.g3grp_l.to_numpy(), False)

In [ ]:
eco.loc[:,'groupra'] = tmpra
eco.loc[:,'groupdec'] = tmpdec
eco.loc[:,'groupcz'] = tmpcz
tmp = eco.groupby('g3grp_l').first()

In [ ]:
plt.figure()
sc=plt.scatter(tmp.g3grpradeg_l, tmp.groupra - tmp.g3grpradeg_l, c=tmp.g3logmh_l, s=1, cmap='rainbow_r')
plt.colorbar(sc, label='Halo Mass')
plt.xlabel("H23 Group RA [deg]")
plt.ylabel("Probabilistic Group RA - H23 Group RA [deg]")
plt.gca().set_facecolor('gray')
#plt.ylim(-0.1,0.1)
print(np.median(np.abs(tmp.groupra - tmp.g3grpradeg_l)) * 3600, 'arcsec')
plt.show()

In [ ]:
plt.figure()
sc=plt.scatter(tmp.g3grpdedeg_l, tmp.groupdec - tmp.g3grpdedeg_l, c=tmp.g3logmh_l, s=1, cmap='rainbow_r')
plt.colorbar(sc, label='Halo Mass')
plt.xlabel("H23 Group Dec [deg]")
plt.ylabel("Probabilistic Group Dec - H23 Group Dec [deg]")
plt.gca().set_facecolor('gray')
#plt.ylim(-0.1,0.1)
print(np.median(np.abs(tmp.groupdec - tmp.g3grpdedeg_l))*3600)
plt.show()

In [ ]:
plt.figure()
sc=plt.scatter(tmp.g3grpcz_l, tmp.groupcz - tmp.g3grpcz_l, c=tmp.g3logmh_l, s=1, cmap='rainbow_r')
plt.colorbar(sc, label='Halo Mass')
plt.xlabel("H23 Group cz [deg]")
plt.ylabel("Probabilistic Group cz - H23 Group cz [deg]")
plt.gca().set_facecolor('gray')

print(np.median(np.abs(tmp.groupcz - tmp.g3grpcz_l)))
plt.show()

# Test Accuracy against Error-Weighted Group Center

In [ ]:
def weighted_group_skycoords(galaxyra, galaxydec, galaxycz, galaxyczerr, galaxygrpid):
    """
    -----
    Obtain a list of group centers (RA/Dec/cz) given a list of galaxy coordinates (equatorial)
    and their corresponding group ID numbers.
    
    Inputs (all same length)
       galaxyra : 1D iterable,  list of galaxy RA values in decimal degrees
       galaxydec : 1D iterable, list of galaxy dec values in decimal degrees
       galaxycz : 1D iterable, list of galaxy cz values in km/s
       galaxygrpid : 1D iterable, group ID number for every galaxy in previous arguments.
    
    Outputs (all shape match `galaxyra`)
       groupra : RA in decimal degrees of galaxy i's group center.
       groupdec : Declination in decimal degrees of galaxy i's group center.
       groupcz : Redshift velocity in km/s of galaxy i's group center.
    
    Note: the FoF code of AA Berlind uses theta_i = declination, with theta_cen = 
    the central declination. This version uses theta_i = pi/2-dec, with some trig functions
    changed so that the output *matches* that of Berlind's FoF code (my "deccen" is the same as
    his "thetacen", to be exact.)
    -----
    """
    # Prepare cartesian coordinates of input galaxies
    ngalaxies = len(galaxyra)
    galaxyphi = galaxyra * np.pi/180.
    galaxytheta = np.pi/2. - galaxydec*np.pi/180.
    galaxyx = np.sin(galaxytheta)*np.cos(galaxyphi)
    galaxyy = np.sin(galaxytheta)*np.sin(galaxyphi)
    galaxyz = np.cos(galaxytheta)
    # Prepare output arrays
    uniqidnumbers = np.unique(galaxygrpid)
    groupra = np.zeros(ngalaxies)
    groupdec = np.zeros(ngalaxies)
    groupcz = np.zeros(ngalaxies)
    for i,uid in enumerate(uniqidnumbers):
        sel=np.where(galaxygrpid==uid)
        #nmembers = len(galaxygrpid[sel])
        denom = np.sum(galaxyczerr[sel])
        xcen=np.sum(galaxyczerr[sel]*galaxycz[sel]*galaxyx[sel])/denom
        ycen=np.sum(galaxyczerr[sel]*galaxycz[sel]*galaxyy[sel])/denom
        zcen=np.sum(galaxyczerr[sel]*galaxycz[sel]*galaxyz[sel])/denom
        czcen = np.sqrt(xcen**2 + ycen**2 + zcen**2)
        deccen = np.arcsin(zcen/czcen)*180.0/np.pi # degrees
        if (ycen >=0 and xcen >=0):
            phicor = 0.0
        elif (ycen < 0 and xcen < 0):
            phicor = 180.0
        elif (ycen >= 0 and xcen < 0):
            phicor = 180.0
        elif (ycen < 0 and xcen >=0):
            phicor = 360.0
        elif (xcen==0 and ycen==0):
            print("Warning: xcen=0 and ycen=0 for group {}".format(galaxygrpid[i]))
        # set up phicorrection and return phicen.
        racen=np.arctan(ycen/xcen)*(180/np.pi)+phicor # in degrees
        # set values at each element in the array that belongs to the group under iteration
        groupra[sel] = racen # in degrees
        groupdec[sel] = deccen # in degrees
        groupcz[sel] = czcen
    return groupra, groupdec, groupcz

In [ ]:
weightedgrpcenter=weighted_group_skycoords(eco.radeg.to_numpy(), eco.dedeg.to_numpy(), eco.degradedcz.to_numpy(), eco.degradedczerr.to_numpy(), eco.g3grp_l.to_numpy())

In [ ]:
eco.loc[:,'weightedgrpra'] = weightedgrpcenter[0]
eco.loc[:,'weightedgrpdec'] = weightedgrpcenter[1]
eco.loc[:,'weightedgrpcz'] = weightedgrpcenter[2]

In [ ]:
df = eco.groupby('g3grp_l').first()

In [ ]:
perr_weighted_ra = (df.weightedgrpra - df.g3grpradeg_l)/df.g3grpradeg_l
perr_weighted_dec = (df.weightedgrpdec - df.g3grpdedeg_l)/df.g3grpdedeg_l
perr_weighted_cz = (df.weightedgrpcz - df.g3grpcz_l)/df.g3grpcz_l

perr_prob_ra = (df.groupra - df.g3grpradeg_l)/df.g3grpradeg_l
perr_prob_dec = (df.groupdec - df.g3grpdedeg_l)/df.g3grpdedeg_l
perr_prob_cz = (df.groupcz - df.g3grpcz_l)/df.g3grpcz_l

In [ ]:
fig, axs = plt.subplots(ncols=3, figsize=(12,4))

histkwargs = {'histtype':'step', 'linewidth':2}

rabins = np.arange(-1,1,0.05)
axs[0].hist(perr_weighted_ra, bins=rabins, label='Error-Weighted')
axs[0].hist(perr_prob_ra, bins=rabins, label='Probabilistic', **histkwargs)
axs[0].set_yscale('log')
axs[0].set_xlabel("Fractional Error in RA")
axs[0].legend(loc='best',fontsize=10)
axs[0].annotate('loc = {:0.1E}'.format(biweight_location(perr_weighted_ra)), xy=(0.1,700),color='tab:blue',fontsize=11)
axs[0].annotate('scale = {:0.1E}'.format(biweight_scale(perr_weighted_ra)), xy=(0.1,500),color='tab:blue',fontsize=11)
axs[0].annotate('loc = {:0.1E}'.format(biweight_location(perr_prob_ra)), xy=(0.1,300),color='tab:orange',fontsize=11)
axs[0].annotate('scale = {:0.1E}'.format(biweight_scale(perr_prob_ra)), xy=(0.1,200),color='tab:orange',fontsize=11)


axs[1].hist(perr_weighted_dec,bins=rabins)
axs[1].hist(perr_prob_dec, bins=rabins, **histkwargs)
axs[1].set_yscale('log')
axs[1].set_xlabel("Fractional Error in Group Dec")
axs[1].annotate('loc = {:0.1E}'.format(biweight_location(perr_weighted_dec)), xy=(0.1,700),color='tab:blue',fontsize=11)
axs[1].annotate('scale = {:0.1E}'.format(biweight_scale(perr_weighted_dec)), xy=(0.1,500),color='tab:blue',fontsize=11)
axs[1].annotate('loc = {:0.1E}'.format(biweight_location(perr_prob_dec)), xy=(0.1,300),color='tab:orange',fontsize=11)
axs[1].annotate('scale = {:0.1E}'.format(biweight_scale(perr_prob_dec)), xy=(0.1,200),color='tab:orange',fontsize=11)

czbins = np.arange(-1,1,0.05)
axs[2].hist(perr_weighted_cz, bins=czbins)
axs[2].hist(perr_prob_cz, bins=czbins, **histkwargs)
axs[2].set_yscale('log')
axs[2].set_xlabel("Fractional Error in Group cz")
axs[2].annotate('loc = {:0.1E}'.format(biweight_location(perr_weighted_cz)), xy=(0.1,700),color='tab:blue',fontsize=11)
axs[2].annotate('scale = {:0.1E}'.format(biweight_scale(perr_weighted_cz)), xy=(0.1,500),color='tab:blue',fontsize=11)
axs[2].annotate('loc = {:0.1E}'.format(biweight_location(perr_prob_cz)), xy=(0.1,300),color='tab:orange',fontsize=11)
axs[2].annotate('scale = {:0.1E}'.format(biweight_scale(perr_prob_cz)), xy=(0.1,200),color='tab:orange',fontsize=11)

for ii in range(0,len(axs)):
    axs[ii].set_ylim(0.7,1200)

plt.show()

In [ ]:
df[['g3grpradeg_l','groupra','weightedgrpra']]

In [ ]:
df[['g3grpcz_l','groupcz','weightedgrpcz']]

# Make Plot for Paper

In [ ]:
from matplotlib.ticker import MaxNLocator
from matplotlib import rcParams
from seaborn import kdeplot
rcParams['axes.labelsize'] = 9
rcParams['xtick.labelsize'] = 9
rcParams['ytick.labelsize'] = 9
rcParams['legend.fontsize'] = 9
rcParams['font.family'] = 'sans-serif'
#rcParams['font.sans-serif'] = ['Helvetica']
#rcParams['text.usetex'] = True
rcParams['grid.color'] = 'k'
rcParams['grid.linewidth'] = 0.2
my_locator = MaxNLocator(6)
singlecolsize = (3.3522420091324205, 2.0717995001590714)
doublecolsize = (7.500005949910059, 4.3880449973709)
from scipy.integrate import simpson as simps

In [ ]:
def plot_prob_group_skycoords(galaxyra, galaxydec, galaxycz, galaxyczerr, galaxygrpid):
    """
    -----
    Obtain a list of group centers (RA/Dec/cz) given a list of galaxy coordinates (equatorial)
    and their corresponding group ID numbers.
    
    Inputs (all same length)
       galaxyra : 1D iterable,  list of galaxy RA values in decimal degrees
       galaxydec : 1D iterable, list of galaxy dec values in decimal degrees
       galaxycz : 1D iterable, list of galaxy cz values in km/s
       galaxyczerr : 1D iterable, list of galaxy cz 
       galaxygrpid : 1D iterable, group ID number for every galaxy in previous arguments.
    
    Outputs (all shape match `galaxyra`)
       groupra : RA in decimal degrees of galaxy i's group center.
       groupdec : Declination in decimal degrees of galaxy i's group center.
       groupcz : Redshift velocity in km/s of galaxy i's group center.
    
    Note: the FoF code of AA Berlind uses theta_i = declination, with theta_cen = 
    the central declination. This version uses theta_i = pi/2-dec, with some trig functions
    changed so that the output *matches* that of Berlind's FoF code (my "deccen" is the same as
    his "thetacen", to be exact.)
    -----
    """
    # Prepare cartesian coordinates of input galaxies
    ngalaxies = len(galaxyra)
    galaxyphi = galaxyra * np.pi/180.
    galaxytheta = np.pi/2. - galaxydec*np.pi/180.
    galaxyx = np.expand_dims((np.sin(galaxytheta)*np.cos(galaxyphi)),axis=1) # equivalent to [:,np.newaxis]
    galaxyy = np.expand_dims((np.sin(galaxytheta)*np.sin(galaxyphi)),axis=1)
    galaxyz = np.expand_dims(((np.cos(galaxytheta))),axis=1)
    # Prepare output arrays
    uniqidnumbers = np.unique(galaxygrpid)
    groupra = np.zeros(ngalaxies)
    groupdec = np.zeros(ngalaxies)
    groupcz = np.zeros(ngalaxies)
    cspeed=3e5
    galaxycz = np.expand_dims(galaxycz,axis=1) / cspeed
    galaxyczerr = np.expand_dims(galaxyczerr,axis=1) / cspeed
    #zzmesh = np.arange(-2,3,1/(cspeed)) # resolution of 1 km/s
    for i,uid in enumerate(uniqidnumbers):
        sel=np.where(galaxygrpid==uid)
        if len(sel[0])==1:
            groupra[sel] = galaxyra[sel]
            groupdec[sel] = galaxydec[sel]
            groupcz[sel] = galaxycz[sel]*cspeed
        else:
            #zzmesh = np.arange(np.min(galaxycz[sel])-5*np.max(galaxyczerr[sel]), np.max(galaxycz[sel])+5*np.max(galaxyczerr[sel]), 1/cspeed)
            xmesh = np.arange(np.min(galaxycz[sel]*galaxyx[sel])-5*np.max(np.abs(galaxyczerr[sel]*galaxyx[sel])), np.max(galaxycz[sel]*galaxyx[sel])+5*np.max(np.abs(galaxyczerr[sel]*galaxyx[sel])), np.min(galaxyczerr[sel])/1000.)
            xcen = get_median_eCDF(xmesh, np.sum(gauss_vectorized(xmesh, galaxycz[sel]*galaxyx[sel], galaxyczerr[sel]*galaxyx[sel]),axis=0))
            ymesh = np.arange(np.min(galaxycz[sel]*galaxyy[sel])-5*np.max(np.abs(galaxyczerr[sel]*galaxyy[sel])), np.max(galaxycz[sel]*galaxyy[sel])+5*np.max(np.abs(galaxyczerr[sel]*galaxyy[sel])), np.min(galaxyczerr[sel])/1000.)
            ycen = get_median_eCDF(ymesh, np.sum(gauss_vectorized(ymesh, galaxycz[sel]*galaxyy[sel], galaxyczerr[sel]*galaxyy[sel]),axis=0))
            zmesh = np.arange(np.min(galaxycz[sel]*galaxyz[sel])-5*np.max(np.abs(galaxyczerr[sel]*galaxyz[sel])), np.max(galaxycz[sel]*galaxyz[sel])+5*np.max(np.abs(galaxyczerr[sel]*galaxyz[sel])), np.min(galaxyczerr[sel])/1000.)
            zcen = get_median_eCDF(zmesh, np.sum(gauss_vectorized(zmesh, galaxycz[sel]*galaxyz[sel], galaxyczerr[sel]*galaxyz[sel]),axis=0))    
            redshiftcen = np.sqrt(xcen*xcen + ycen*ycen + zcen*zcen)
            deccen = np.arcsin(zcen/redshiftcen)*180.0/np.pi # degrees
            if (ycen >=0 and xcen>=0):
                phicor = 0.0
            elif (ycen < 0 and xcen < 0):
                 phicor = 180.0
            elif (ycen >= 0 and xcen < 0):
                phicor = 180.0
            elif (ycen < 0 and xcen >=0):
                phicor = 360.0
            elif (xcen==0 and ycen==0):
                 print("Warning: xcen=0 and ycen=0 for group ", uid)
            racen=np.arctan(ycen/xcen)*(180/np.pi)+phicor # in degrees
            groupra[sel] = racen # in degrees
            groupdec[sel] = deccen # in degrees
            groupcz[sel] = cspeed * redshiftcen
            
            fig,axs=plt.subplots(ncols=3,nrows=2,figsize=(doublecolsize[0],doublecolsize[1]*0.9))
            x_pdf = np.sum(gauss_vectorized(xmesh, galaxycz[sel]*galaxyx[sel], galaxyczerr[sel]*galaxyx[sel]),axis=0)
            x_pdf /= simps(x_pdf, xmesh)
            axs[0][0].plot(xmesh, x_pdf / np.sum(x_pdf))
            x_cdf = np.cumsum(x_pdf)/np.sum(x_pdf)
            axs[1][0].step(xmesh, x_cdf)
            axs[1][0].axvline(xmesh[np.argmin(np.abs(x_cdf-0.5))], color='k', linestyle='dashed')
            
            y_pdf = np.sum(gauss_vectorized(ymesh, galaxycz[sel]*galaxyy[sel], galaxyczerr[sel]*galaxyy[sel]),axis=0)
            y_pdf /= simps(y_pdf, ymesh)
            axs[0][1].plot(ymesh, y_pdf / np.sum(y_pdf))
            y_cdf = np.cumsum(y_pdf)/np.sum(y_pdf)
            axs[1][1].step(ymesh, y_cdf)
            axs[1][1].axvline(ymesh[np.argmin(np.abs(y_cdf-0.5))], color='k', linestyle='dashed')
            
            z_pdf = np.sum(gauss_vectorized(zmesh, galaxycz[sel]*galaxyz[sel], galaxyczerr[sel]*galaxyz[sel]),axis=0)
            z_pdf /= simps(z_pdf, zmesh)
            axs[0][2].plot(zmesh, z_pdf / np.sum(z_pdf))
            z_cdf = np.cumsum(z_pdf)/np.sum(z_pdf)
            axs[1][2].step(zmesh, z_cdf)
            axs[1][2].axvline(zmesh[np.argmin(np.abs(z_cdf-0.5))], color='k', linestyle='dashed')
            axs[0][2].set_xlim(np.min(zmesh),np.max(zmesh))
            
            axs[0][0].set_ylabel("Group Center PDF")
            axs[1][0].set_ylabel("Empirical CDF")
            axs[1][0].set_xlabel(r'$z \cdot w$')
            axs[1][1].set_xlabel(r'$z \cdot x$')
            axs[1][2].set_xlabel(r'$z \cdot y$')
            for ii in range(0,3):
                axs[0][ii].set_ylim(0,0.0006)
                xticks=np.linspace(axs[0][ii].lines[0].get_xdata().min(), axs[0][ii].lines[0].get_xdata().max(), 3)
                axs[0][ii].set_xticks(xticks,labels=['{:0.4f}'.format(xt) for xt in xticks])
                axs[1][ii].set_xticks(xticks,labels=['{:0.4f}'.format(xt) for xt in xticks])
                axs[0][ii].set_xlim(xticks.min(),xticks.max())
                axs[1][ii].set_xlim(xticks.min(),xticks.max())
                
                axs[1][ii].axhline(0.5,color='gray',alpha=0.7,zorder=0)
                
            plt.tight_layout()
            plt.savefig("../figures/groupcenterillustration.pdf",dpi=300)
            plt.show()
    return groupra, groupdec, groupcz

In [ ]:
grp92 = eco[eco.g3grp_l==92]

In [ ]:
plot_prob_group_skycoords(grp92.radeg, grp92.dedeg, grp92.cz, grp92.cz*0+35, grp92.g3grp_l)

In [ ]:
grp92[['g3grpradeg_l','g3grpdedeg_l','g3grpcz_l']]